# Who Wants to Be a PoliMillionaire? — NLP Group Assignment 2025/26

| | |
|---|---|
| **Group members** | _Name 1_ — `email1@mail.polimi.it` · _Name 2_ — `email2@mail.polimi.it` · _Name 3_ — `email3@mail.polimi.it` |
| **Video link** | _paste URL here_ |
| **Default model** | Qwen 2.5 7B-Instruct (4-bit NF4) |
| **Default strategy** | Tiered: BaselineLLM → RAG → Ensemble · Maths → Agent |

This notebook is the **experimentation workbench** for the assignment.
The implementation lives in the `polimibot` Python package — this
notebook only configures, runs, compares, and saves results.

**How to use this notebook:**

1. Run **Section 0 — Setup** once per Colab session.
2. Edit knobs in **Section 1 — Configure** to choose your strategy.
3. Run **Section 2 — Run** to evaluate it and save a report.
4. Repeat 1→2 for as many configurations as you want.
5. Use **Section 3 — Compare** to read every saved report into a leaderboard.
6. **Section 4 — Save** writes consolidated CSVs for your final write-up.

Switching strategies should not require touching anything outside Section 1.

---

## 0. Setup

Install the package, import helpers, log in to the game server. Run this section once per Colab session — re-running it is harmless but slow.

> **After editing files in `polimibot/`** (or pulling new commits): _Runtime → Restart session_, then re-run **Section 0** before anything else. Even with `pip install -e .`, classes already imported in this kernel stay cached — restarting is the only reliable way to pick up the new code.

### 0.0 Mount Google Drive (Colab only)

Two modes — controlled by the `clone` flag:

| `clone` | Code source | `data/` location |
|---------|-------------|-----------------|
| `True`  | Fresh git clone into `/content/PoliMillionaire/` (fast, not throttled) | Symlinked → Drive so run logs, gold set, and the RAG index survive session restarts |
| `False` | Work directly from the Drive copy | Native `data/` inside the Drive project folder |

When `clone=True`, a symlink `data/ → <drive_path>/data/` is created once and
reused on subsequent sessions — no data loss on kernel restart.

In [ ]:
# Mount Google Drive and set up the project. Two modes: clone or drive-direct.
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_PROJECT = "/content/drive/MyDrive/Colab Notebooks/Polimillionaire"

clone = True  # ← flip to False to work entirely from the Drive copy

if clone:
    # Clone fresh code from GitHub into fast local filesystem.
    REPO_URL = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/m-ebrahimzadeh/PoliMillionaire.git"
    if not os.path.exists('/content/PoliMillionaire'):
        print("Cloning repo…")
        !git clone {REPO_URL} /content/PoliMillionaire
    else:
        print("Repo already cloned — skipping clone.")
    %cd /content/PoliMillionaire

    # Symlink data/ → Drive so runs, gold sets, and the RAG index persist
    # across session restarts. Created once; reused on every subsequent run.
    import shutil as _shutil
    os.makedirs(f"{DRIVE_PROJECT}/data", exist_ok=True)
    if os.path.isdir('data') and not os.path.islink('data'):
        _shutil.rmtree('data')   # remove the empty placeholder from the fresh clone
    if not os.path.exists('data'):
        os.symlink(f"{DRIVE_PROJECT}/data", 'data')
    print(f'data/ → {os.readlink("data")}  (persistent on Drive)')
else:
    %cd "{DRIVE_PROJECT}"
    print(f'Working directly from Drive: {DRIVE_PROJECT}')

### 0.1 Install

In [ ]:
# Install the project as an editable package, this cell does. Once per session, run it.
%pip install -q -e .
%pip install -q "transformers>=4.44" "accelerate>=0.33" "bitsandbytes>=0.43"
%pip install -q "faiss-cpu>=1.7" "sentence-transformers>=2.7" "wikipedia>=1.4"
%pip install -q matplotlib pandas

### 0.2 Imports and helpers

In [ ]:
# All imports, here once. Reach further than this cell, the notebook should not.
from __future__ import annotations

import gc
import json
import os
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Polimibot — the audited package.
from polimibot import (
    PATHS, RUNTIME, update_runtime, CATEGORIES, Category,
    GameQuestion, GameAdapter, AnswerOutcome, SessionRecord,
    Strategy, StrategyInput, StrategyOutput,
    GameResult, play_game,
    RunLogger, NullLogger, load_jsonl,
)
from polimibot.models.mock import MockLLM
from polimibot.prompts.templates import PromptStyle
from polimibot.strategies.llm_baseline import BaselineLLMStrategy
from polimibot.strategies.rag_strategy import RAGStrategy
from polimibot.strategies.tool_strategy import ToolStrategy
from polimibot.strategies.agent_strategy import AgentStrategy
from polimibot.strategies.ensemble_strategy import EnsembleStrategy
from polimibot.strategies.tiered_strategy import TieredStrategy, TierBreakpoints
from polimibot.tools.maths_tool import MathsTool
from polimibot.eval.evaluator import evaluate_strategy, EvalReport
from polimibot.eval.gold_set import GoldSet, load_gold_set, harvest_gold_set, save_gold_set
from polimibot.eval.report_io import save_report, model_slug
from polimibot.eval.calibration import calibration_from_runs, plot_calibration

# Make scripts/_session.py importable for the live-game session helper.
sys.path.insert(0, str(PATHS.project_root / 'scripts'))
from _session import play_session

PATHS.ensure()
print(f'project root : {PATHS.project_root}')
print(f'data/         : {PATHS.data_dir}')
print(f'API URL       : {RUNTIME.api_url}')

In [ ]:
# Helpers for VRAM hygiene and disk persistence — one place, defined they are.

def unload_llm(llm) -> None:
    """Free the LLM's GPU references. Call before loading another model."""
    if llm is None:
        return
    for attr in ('_model', '_tokenizer'):
        if hasattr(llm, attr):
            try:
                delattr(llm, attr)
            except Exception:
                pass


def clear_vram() -> None:
    """Release cached CUDA memory. Pair with unload_llm before any new load."""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except ImportError:
        pass


RESULTS_DIR = PATHS.project_root / 'data' / 'results'


def save_results(df: pd.DataFrame, name: str) -> Path:
    """Persist a dataframe to data/results/{name}.csv. Crash-safety, this gives."""
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    path = RESULTS_DIR / f'{name}.csv'
    df.to_csv(path, index=False)
    print(f'Saved → {path}')
    return path


def latest_run_log(strategy_substring: str = '') -> Optional[Path]:
    """Most recent JSONL run log under data/runs/, optionally filtered by name."""
    runs = sorted(PATHS.runs_dir.glob('run_*.jsonl'))
    if strategy_substring:
        runs = [r for r in runs if strategy_substring in r.name]
    return runs[-1] if runs else None


print('helpers ready ✓')

### 0.3 Log in (live games only — skip for offline eval)

In [ ]:
# Credentials, from env we read. Set in Colab: os.environ["POLIMI_USER"] = "..."
# Skip this cell entirely if only the offline gold set you intend to evaluate.
from millionaire_client import MillionaireClient

POLIMI_USER = os.environ.get('POLIMI_USER', '')
POLIMI_PASS = os.environ.get('POLIMI_PASS', '')

client = None
if POLIMI_USER and POLIMI_PASS:
    client = MillionaireClient(RUNTIME.api_url)
    client.login(POLIMI_USER, POLIMI_PASS)
    print(f'Logged in as {POLIMI_USER} → {RUNTIME.api_url}')
else:
    print('No credentials in env — live games disabled. Offline eval will still work.')

### 0.4 Build / top-up the RAG knowledge index

Run this section **once** before the first RAG experiment — or whenever you want to refresh the Wikipedia corpus. The result is a FAISS index (+ BM25 sidecar) stored under `data/cache/knowledge.*`.

| Knob | Meaning |
|---|---|
| `REBUILD_INDEX` | Set `True` to force a full rebuild even if the index already exists |
| `INDEX_REFETCH` | Set `True` to re-download Wikipedia articles (ignored if `REBUILD_INDEX=False` and index exists) |
| `INDEX_CATEGORIES` | `None` → all four categories; or e.g. `['history', 'science']` to build a partial index |
| `INDEX_CHUNK_SIZE` / `INDEX_OVERLAP` | Sentence-aware chunking parameters (default 300/50) |
| `INDEX_SKIP_BM25` | Set `True` to skip the BM25 sidecar (dense-only build — faster but disables hybrid retrieval) |

Skipped automatically when the index already exists and `REBUILD_INDEX=False` — safe to leave in the run-all path.

In [ ]:
# ─── RAG index knobs. ────────────────────────────────────────────────
# RAG_INDEX_PATH is also set in Section 1.1; defined here so this cell runs
# standalone without requiring the Knobs cell to be executed first.
RAG_INDEX_PATH     = PATHS.cache_dir / 'knowledge'
REBUILD_INDEX      = False        # True  → rebuild even if index already exists
INDEX_REFETCH      = False        # True  → re-download Wikipedia (slow, ~5–10 min)
INDEX_CATEGORIES   = None         # None  → all four; or e.g. ['history', 'science']
INDEX_CHUNK_SIZE   = 300          # words per chunk (sentence-boundary-aware)
INDEX_OVERLAP      = 50           # word overlap between adjacent chunks
INDEX_SKIP_BM25    = False        # True  → skip BM25 sidecar (dense-only, faster)
# ─────────────────────────────────────────────────────────────────────

_index_faiss = RAG_INDEX_PATH.with_suffix('.faiss')
_index_exists = _index_faiss.exists()

if _index_exists and not REBUILD_INDEX:
    print(f'Index already exists: {_index_faiss}')
    print('Set REBUILD_INDEX=True to force a rebuild.')
else:
    import time as _time
    from polimibot.config import Category as _Category
    from polimibot.rag.corpus import (
        fetch_articles, load_raw_corpus, save_raw_corpus, clean_wikipedia_text,
        CORPUS_VERSION, CLEANUP_VERSION,
    )
    from polimibot.rag.chunker import CHUNKER_VERSION, chunk_text as _chunk_text
    from polimibot.rag.embedder import Embedder as _Embedder, EmbedderSpec as _EmbedderSpec
    from polimibot.rag.index import FAISSIndex as _FAISSIndex
    from polimibot.rag.bm25 import BM25Index as _BM25Index
    from dataclasses import replace as _replace

    _corpus_path = PATHS.cache_dir / 'corpus.jsonl'
    PATHS.ensure()

    # ── Step 1: corpus ───────────────────────────────────────────────
    _cats = [_Category(c) for c in INDEX_CATEGORIES] if INDEX_CATEGORIES else None

    if _corpus_path.exists() and not INDEX_REFETCH:
        print(f'Loading cached corpus from {_corpus_path}…')
        _articles = load_raw_corpus(_corpus_path)
        if _cats:
            _articles = [a for a in _articles if a.category in _cats]
        _articles = [_replace(a, text=clean_wikipedia_text(a.text)) for a in _articles]
    else:
        print('Fetching articles from Wikipedia (this takes several minutes)…')
        _articles = fetch_articles(categories=_cats, verbose=True)
        save_raw_corpus(_articles, _corpus_path)

    if not _articles:
        raise RuntimeError('No articles fetched — check your network connection and INDEX_CATEGORIES.')

    # ── Step 2: chunk ────────────────────────────────────────────────
    print(f'\nChunking {len(_articles)} articles (size={INDEX_CHUNK_SIZE}, overlap={INDEX_OVERLAP})…')
    _all_chunks = []
    for _art in _articles:
        _all_chunks.extend(_chunk_text(
            _art.text, source=_art.title,
            chunk_size=INDEX_CHUNK_SIZE, overlap=INDEX_OVERLAP,
            category=_art.category.value,
        ))
    print(f'  → {len(_all_chunks)} chunks '
          f'(avg {len(_all_chunks) // max(len(_articles), 1)} per article)')

    # ── Step 3: embed ────────────────────────────────────────────────
    print('\nLoading embedding model (bge-small-en-v1.5)…')
    _spec = _EmbedderSpec()        # uses default: BAAI/bge-small-en-v1.5
    _embedder = _Embedder(_spec)
    print(f'  dim={_embedder.dim}  batch={_spec.batch_size}')

    print('Embedding passages…')
    _t0 = _time.monotonic()
    _embeddings = _embedder.encode_passage([c.text for c in _all_chunks])
    print(f'  → done in {_time.monotonic() - _t0:.1f}s')

    # ── Step 4: FAISS index ──────────────────────────────────────────
    _idx = _FAISSIndex(dim=_embedder.dim)
    _idx.add(_all_chunks, _embeddings)
    _idx.save(RAG_INDEX_PATH, manifest={
        'embedder_model_name':     _spec.model_name,
        'embedder_dim':            _embedder.dim,
        'embedder_query_prefix':   _spec.query_prefix,
        'embedder_passage_prefix': _spec.passage_prefix,
        'normalize':               _spec.normalize,
        'chunk_size':              INDEX_CHUNK_SIZE,
        'chunk_overlap':           INDEX_OVERLAP,
        'chunker_version':         CHUNKER_VERSION,
        'corpus_version':          CORPUS_VERSION,
        'n_articles':              len(_articles),
        'text_cleanup_version':    CLEANUP_VERSION,
        'categories':              sorted({a.category.value for a in _articles}),
    })
    print(f'\n✓  FAISS index saved → {RAG_INDEX_PATH}.faiss')
    print(f'   {_idx.n_chunks} chunks | dim={_embedder.dim} | model={_spec.model_name}')

    # ── Step 5: BM25 sidecar ─────────────────────────────────────────
    if not INDEX_SKIP_BM25:
        print('\nBuilding BM25 sidecar…')
        _t0 = _time.monotonic()
        _bm25 = _BM25Index(_all_chunks)
        _bm25.save(RAG_INDEX_PATH)
        print(f'   built in {_time.monotonic() - _t0:.1f}s  →  {RAG_INDEX_PATH}.bm25.jsonl')
    else:
        print('BM25 sidecar skipped (INDEX_SKIP_BM25=True).')

    print('\nIndex build complete. Re-run Section 1.3 to attach the new index to the retriever.')

> **Setup observations** — _fill in after running this section._
>
> _What worked, what didn't, what would you try next?_

---

## 1. Configure

Pick a strategy by editing the knobs below. **One variable per concept** — change `MODEL_ID`, `PROMPT_STYLE`, `USE_RAG`, etc. independently. The next cell consumes these knobs and constructs a single `strategy` object.

**Common experiments:**
- _Plain LLM baseline_ → `USE_MOCK=False`, others default.
- _Few-shot vs zero-shot_ → toggle `PROMPT_STYLE`.
- _RAG ablation_ → flip `USE_RAG`.
- _Tools-only on maths_ → `USE_MATHS_TOOL=True`, `USE_RAG=False`.
- _Full tiered system_ → `USE_TIERED=True`.
- _Smoke test on CPU_ → `USE_MOCK=True`.

### 1.1 Knobs

In [ ]:
# ─── Knobs. The only cell, edit you should, between experiments. ───

# Model
USE_MOCK            = False                              # CPU smoke-test mode (no GPU)
MODEL_ID            = 'Qwen/Qwen2.5-7B-Instruct'         # any HF causal LM
LOAD_IN_4BIT        = True                               # NF4 quantisation; False on CPU
TRUST_REMOTE_CODE   = False                              # set True for some Phi/DeepSeek/Yi releases

# Prompt
PROMPT_STYLE        = PromptStyle.ZERO_SHOT              # ZERO_SHOT, FEW_SHOT, *_COT
USE_SCORE_OPTIONS   = True                               # logit-scoring (False forces free generation)

# Generation budgets (only used when USE_SCORE_OPTIONS=False)
DIRECT_MAX_NEW_TOKENS = 16                               # ZERO_SHOT / FEW_SHOT — small, just "Answer: X"
COT_MAX_NEW_TOKENS    = 256                              # *_COT — room for the reasoning trace
STOP_STRINGS          = None                             # None → boxed-stops on CoT; or pass a list

# Per-question deadline (server gives 30s; 25s leaves margin for submit roundtrip)
HARD_CUTOFF_SECONDS = 25.0                               # strategy must return by this
update_runtime(hard_cutoff_seconds=HARD_CUTOFF_SECONDS)  # rebinds RUNTIME singleton, this does

# Strategy composition (highest USE_TIERED wins; otherwise stack from baseline up)
USE_RAG             = False                              # RAG over Wikipedia
USE_MATHS_TOOL      = False                              # deterministic arithmetic on maths Qs
USE_AGENT_FOR_MATHS = False                              # ReAct agent on maths Qs
USE_ENSEMBLE        = False                              # weighted-prob fusion of baseline + RAG
USE_TIERED          = False                              # full hybrid: tier-by-level with all the above

# Tier knobs (only used when USE_TIERED=True)
TIER_EASY_MAX       = 5                                  # levels 1..5  → easy strategy
TIER_MEDIUM_MAX     = 10                                 # levels 6..10 → medium strategy
ESCALATION_THRESHOLD = None                              # e.g. 0.15 to escalate on low margin

# Retrieval (only used when USE_RAG / USE_ENSEMBLE / USE_TIERED)
RAG_K                  = 3                               # passages per question
RAG_INDEX_PATH         = PATHS.cache_dir / 'knowledge'   # build with scripts/build_rag_index.py
RAG_USE_CATEGORY_FILTER = True                           # restrict retrieval to inp.category chunks
RAG_USE_SCORE_OPTIONS  = True                            # logit-scoring (False = free generation; required for CoT/ELIMINATION)
RAG_MAX_PASSAGE_CHARS  = 800                             # per-passage truncation
RAG_MAX_TOTAL_CHARS    = 2400                            # joined context budget

# Reranker (cross-encoder over the dense pool — precision win, +~30 ms/query)
RAG_USE_RERANKER       = False                           # set True to load + use
RERANKER_MODEL         = 'BAAI/bge-reranker-base'        # trivia-friendly, ~100 MB
RERANK_OVERSEARCH      = 5                               # dense pool size = k × this

# Hybrid + multi-query (lexical complement + per-option queries, both via RRF)
RAG_USE_HYBRID         = False                           # dense + BM25 fused per query
RAG_USE_MULTI_QUERY    = True                            # 1 question + 4 per-option queries (default on per audit §3)

# Path-aware min_score gates (audit §4): calibrate each threshold on its own
# score scale — never apply a cosine threshold to an RRF or reranker score.
RAG_MIN_SCORE          = None                            # dense-only cosine ∈ [-1,1]; e.g. 0.30
RAG_MIN_SCORE_RRF      = None                            # hybrid RRF ∈ ~0–0.03;      e.g. 0.010
RAG_MIN_SCORE_RERANK   = None                            # cross-encoder logit;        e.g. -2.0

# Live-search fallback + self-growing index (fires only when offline RAG is gated)
# When USE_LIVE_FALLBACK=True and the top offline retrieval score is below the
# min_score threshold, a real-time Wikipedia API query is fired instead of
# degrading to a bare-LLM prompt.  Confirmed-correct articles are added to
# the offline index permanently so future questions benefit.
USE_LIVE_FALLBACK      = False                           # True  → enable live Wikipedia fallback
LIVE_SEARCH_TIMEOUT    = 5.0                             # hard wall-clock limit per live query (s)
LIVE_MAX_ARTICLES      = 2                               # max Wikipedia articles per live query
LEARN_FROM_CORRECT     = True                            # grow the index from confirmed-correct answers

# Eval
N_EVAL_QUESTIONS    = None                               # None = all gold items; int = first-N slice

print(f'mock={USE_MOCK}  model={MODEL_ID}  style={PROMPT_STYLE.value}')
print(f'rag={USE_RAG}  maths_tool={USE_MATHS_TOOL}  agent={USE_AGENT_FOR_MATHS}')
print(f'ensemble={USE_ENSEMBLE}  tiered={USE_TIERED}')
print(f'hard_cutoff={HARD_CUTOFF_SECONDS}s  direct_max_new_tokens={DIRECT_MAX_NEW_TOKENS}  cot_max_new_tokens={COT_MAX_NEW_TOKENS}')

### 1.2 Build the LLM (heavy — runs once per model change)

In [ ]:
# Try to reuse an already-loaded LLM. If the model id changed, free GPU memory first.
prev_model_id = globals().get('LOADED_MODEL_ID', None)
if 'llm' in globals() and prev_model_id != MODEL_ID:
    print(f'Model changed ({prev_model_id} → {MODEL_ID}). Unloading previous LLM…')
    unload_llm(llm)
    clear_vram()
    del llm

if 'llm' not in globals():
    if USE_MOCK:
        llm = MockLLM(name='mock', correctness=0.65)
        print('MockLLM ready (CPU, no GPU needed)')
    else:
        from polimibot.models.llm import LLM, LLMSpec
        spec = LLMSpec(
            model_id=MODEL_ID,
            load_in_4bit=LOAD_IN_4BIT,
            trust_remote_code=TRUST_REMOTE_CODE,
        )
        llm = LLM.load(spec)
    LOADED_MODEL_ID = MODEL_ID
else:
    print(f'Reusing already-loaded LLM: {llm.name}')

### 1.3 Build the retriever (only if RAG-related knobs are on)

In [ ]:
# Lazy retriever. Built only when at least one strategy needs it.
# Reranker is loaded once and attached to the retriever — heavy (~100 MB),
# so we cache it across re-runs of Section 1.3 the same way the LLM is cached.
need_retriever = USE_RAG or USE_ENSEMBLE or USE_TIERED

retriever = None
if need_retriever:
    # Cache the reranker model across Section 1 re-runs.
    prev_rer = globals().get('LOADED_RERANKER_MODEL', None)
    if RAG_USE_RERANKER and (
        'reranker_obj' not in globals() or prev_rer != RERANKER_MODEL
    ):
        from polimibot.rag.reranker import CrossEncoderReranker, RerankerSpec
        print(f'Loading cross-encoder reranker: {RERANKER_MODEL} …')
        reranker_obj = CrossEncoderReranker.load(
            RerankerSpec(model_name=RERANKER_MODEL)
        )
        LOADED_RERANKER_MODEL = RERANKER_MODEL
    elif not RAG_USE_RERANKER:
        reranker_obj = None
    else:
        print(f'Reusing already-loaded reranker: {reranker_obj.name}')

    if USE_MOCK:
        # Null retriever for offline smoke tests, this is.
        class _NullRetriever:
            n_chunks = 0
            has_reranker = False
            has_bm25 = False
            def retrieve(self, q, k=3, *, category=None, rerank=False,
                         rerank_oversearch=None, hybrid=False):
                return []
        retriever = _NullRetriever()
        print('NullRetriever (mock mode — no FAISS index needed)')
    else:
        if not RAG_INDEX_PATH.with_suffix('.faiss').exists():
            raise FileNotFoundError(
                f'RAG index missing: {RAG_INDEX_PATH}.faiss\n'
                'Build it first:\n'
                '  python scripts/build_rag_index.py'
            )
        from polimibot.rag.retriever import Retriever
        from polimibot.rag.embedder import EmbedderSpec
        retriever = Retriever.from_saved(
            RAG_INDEX_PATH, embedder_spec=EmbedderSpec(),
        )
        if reranker_obj is not None:
            retriever._reranker = reranker_obj   # late-attach
        # Late-attach BM25 if its sidecar exists (built by --no-bm25=False).
        if RAG_USE_HYBRID:
            from polimibot.rag.bm25 import BM25Index
            bm25_path = RAG_INDEX_PATH.with_suffix('.bm25.jsonl')
            if not bm25_path.exists():
                raise FileNotFoundError(
                    f'BM25 sidecar missing: {bm25_path}\n'
                    'Rebuild the index without --no-bm25:\n'
                    '  python scripts/build_rag_index.py'
                )
            retriever._bm25 = BM25Index.load(RAG_INDEX_PATH)
        rer_tag = f' + reranker {reranker_obj.name}' if reranker_obj else ''
        bm25_tag = f' + BM25 ({retriever._bm25.n_chunks} chunks)' if retriever.has_bm25 else ''
        print(f'Retriever ready: {retriever.n_chunks} chunks indexed{rer_tag}{bm25_tag}')
else:
    print('Retriever not needed for this configuration.')
    reranker_obj = None

### 1.4 Compose the strategy

In [ ]:
# Strategy factory. Knobs in, one Strategy out — the only place composition lives.

baseline = BaselineLLMStrategy(
    llm,
    style=PROMPT_STYLE,
    use_score_options=USE_SCORE_OPTIONS,
    direct_max_new_tokens=DIRECT_MAX_NEW_TOKENS,
    cot_max_new_tokens=COT_MAX_NEW_TOKENS,
    stop_strings=STOP_STRINGS,
)

# ── IndexGrower (self-growing index) ────────────────────────────────────
# Built only when USE_LIVE_FALLBACK=True AND a real (non-mock) retriever is
# available.  The grower buffers live-fetched articles during gameplay; the
# runner confirms and persists them at session end.  For offline eval it is
# unused — the grower.confirm() / flush() calls in runner.py are no-ops when
# grower is None.
_grower = None
if USE_LIVE_FALLBACK and need_retriever and not USE_MOCK:
    from polimibot.rag.index_grower import IndexGrower
    from polimibot.rag.embedder import Embedder, EmbedderSpec
    _embedder_for_grower = Embedder(EmbedderSpec())   # same spec as retriever build
    _grower = IndexGrower(
        retriever, _embedder_for_grower, RAG_INDEX_PATH,
        corpus_path=PATHS.cache_dir / 'corpus.jsonl',
    )
    print(f'IndexGrower ready — will learn from {_grower.n_learned} confirmed articles so far')
elif USE_LIVE_FALLBACK:
    print('IndexGrower: skipped (USE_MOCK=True or no retriever) — live search context-only mode')

# Shared RAGStrategy kwargs — assemble once so all three call sites stay in sync.
_rag_kwargs = dict(
    k=RAG_K,
    style=PROMPT_STYLE,
    use_score_options=RAG_USE_SCORE_OPTIONS,
    use_category_filter=RAG_USE_CATEGORY_FILTER,
    use_reranker=RAG_USE_RERANKER,
    use_hybrid=RAG_USE_HYBRID,
    use_multi_query=RAG_USE_MULTI_QUERY,
    rerank_oversearch=RERANK_OVERSEARCH,
    # Path-aware min_score gates (audit §4): each threshold is calibrated
    # for its own score scale — dense cosine, RRF, or cross-encoder logit.
    min_score=RAG_MIN_SCORE,
    min_score_rrf=RAG_MIN_SCORE_RRF,
    min_score_rerank=RAG_MIN_SCORE_RERANK,
    max_passage_chars=RAG_MAX_PASSAGE_CHARS,
    max_total_chars=RAG_MAX_TOTAL_CHARS,
    # Live-search fallback — fires only when offline retrieval is gated.
    use_live_fallback=USE_LIVE_FALLBACK,
    live_search_timeout=LIVE_SEARCH_TIMEOUT,
    live_max_articles=LIVE_MAX_ARTICLES,
    index_grower=_grower if LEARN_FROM_CORRECT else None,
)

if USE_TIERED:
    rag_arm    = RAGStrategy(llm, retriever, **_rag_kwargs)
    ensemble   = EnsembleStrategy([baseline, rag_arm], weights=[1.0, 1.2])
    maths_arm  = (
        AgentStrategy(llm, max_iterations=3) if USE_AGENT_FOR_MATHS
        else (ToolStrategy([MathsTool()], fallback=baseline) if USE_MATHS_TOOL else None)
    )
    strategy = TieredStrategy(
        easy=baseline, medium=rag_arm, hard=ensemble,
        breakpoints=TierBreakpoints(
            easy_max_level=TIER_EASY_MAX,
            medium_max_level=TIER_MEDIUM_MAX,
        ),
        maths_override=maths_arm,
        escalation_threshold=ESCALATION_THRESHOLD,
    )
elif USE_ENSEMBLE:
    rag_arm  = RAGStrategy(llm, retriever, **_rag_kwargs)
    strategy = EnsembleStrategy([baseline, rag_arm], weights=[1.0, 1.2])
elif USE_AGENT_FOR_MATHS:
    strategy = AgentStrategy(llm, max_iterations=3)
elif USE_MATHS_TOOL:
    strategy = ToolStrategy([MathsTool()], fallback=baseline)
elif USE_RAG:
    strategy = RAGStrategy(llm, retriever, **_rag_kwargs)
else:
    strategy = baseline

# report_id is computed here (Section 1) — Section 2 saves under it, Section 2.6
# uses it as the live-game run_id. Defining it after the strategy means a student
# can run live games (2.6) without first running offline eval (2.2 / 2.3).
mslug      = model_slug(MODEL_ID, mock=USE_MOCK)
short_tag  = strategy.name.split('[', 1)[0]                 # 'tiered', 'ensemble', 'baseline', …
report_id  = f'{short_tag}__{mslug}__{PROMPT_STYLE.value}'

print(f'Strategy:\n  {strategy.name}')
print(f'report_id: {report_id}')
if USE_LIVE_FALLBACK:
    print(f'Live-search fallback: ENABLED (timeout={LIVE_SEARCH_TIMEOUT}s, max_articles={LIVE_MAX_ARTICLES})')
    print(f'Index growing: {"ENABLED (learn from correct answers)" if LEARN_FROM_CORRECT else "DISABLED"}')

### 1.5 Strategy smoke-test — inspect the prompt & output

Run a single question through the strategy to verify the pipeline end-to-end and
see exactly what is delivered to the model.

1. **Bare prompt** — the raw messages list (system + user turn) that the LLM receives
   when no RAG context is available.
2. **RAG-augmented prompt** — the same structure with retrieved passages injected
   (only shown when the retriever is active).
3. **Strategy output** — the chosen answer, confidence score, rationale, and any
   internal extras (probability margins, retrieval hits, etc.).

In [ ]:
# ─── Single-question smoke test. See what the model actually receives. ───
from polimibot.prompts.templates import build_messages, build_messages_with_context

_test_question = (
    'A farmer wants to know whether a new fertilizer has increased the mean '
    'weight of his apples. With the old fertilizer, the mean weight was 4.0 '
    'ounces per apple. The farmer decides to test H0: μ = 4.0 ounces versus '
    'Ha : μ > 4.0 ounces, at a 5 percent level of significance, where μ = '
    'the mean weight of apples using the new fertilizer. The weights of apples '
    'are approximately normally distributed. The farmer takes a random sample '
    'of 16 apples and computes a mean of 4.3 ounces and a standard deviation '
    'of 0.6 ounces. Which of the following gives the p-value for this test?'
)
_test_options = (
    'P(Z > 2)',
    'P(t > 2) with 15 degrees of freedom',
    'P(t < 2) with 15 degrees of freedom',
    'P(Z < 2)',
)

# ── 1. Bare prompt (no RAG context) ──────────────────────────────────
_bare_msgs = build_messages(_test_question, _test_options,
                            category=None, style=PROMPT_STYLE)
print('═' * 60)
print('BARE PROMPT (no RAG context)')
print('═' * 60)
for _m in _bare_msgs:
    print(f"\n[{_m['role'].upper()}]")
    print(_m['content'])

# ── 2. RAG-augmented prompt (only when retriever is active) ───────────
if retriever is not None and retriever.n_chunks > 0:
    _hits = retriever.retrieve(_test_question, k=RAG_K)
    _ctx  = '\n---\n'.join(h.text[:RAG_MAX_PASSAGE_CHARS] for h in _hits)
    _rag_msgs = build_messages_with_context(
        _test_question, _test_options, _ctx,
        category=None, style=PROMPT_STYLE,
    )
    print('\n' + '═' * 60)
    print(f'RAG-AUGMENTED PROMPT  ({len(_hits)} passage(s) retrieved)')
    print('═' * 60)
    for _m in _rag_msgs:
        print(f"\n[{_m['role'].upper()}]")
        print(_m['content'])
else:
    print('\n(Retriever not active — RAG-augmented prompt not shown.)')

# ── 3. Full strategy output ───────────────────────────────────────────
print('\n' + '═' * 60)
print('STRATEGY OUTPUT')
print('═' * 60)
_out = strategy.answer(StrategyInput(
    question=_test_question, options=_test_options, level=1))
print('rationale :', _out.rationale)
print('chosen    :', _out.chosen_index, '  (A=0, B=1, C=2, D=3)')
print('confidence:', f'{_out.confidence:.2%}')
if _out.extras:
    print('extras    :', _out.extras)

> **Configuration observations** — _fill in after running this section._
>
> _What worked, what didn't, what would you try next?_

---

## 2. Run

Two ways to exercise the strategy:

- **Offline evaluation** on the frozen gold set (fast, deterministic, no network).
- **Live games** against the assignment server (slow, costs API calls, requires login).

The notebook prefers offline. Run live only when you want a fresh row of run-log data, or when you need to top up the gold set.

### 2.0 Build / top-up the gold set from run logs

Harvests confirmed-correct items from all JSONL run logs in `data/runs/` and
merges them into `data/eval/gold_set.jsonl`. Run this after live games so the
freshly confirmed answers are available for offline evaluation.

Safe to re-run at any time — items already present are not duplicated.

In [ ]:
# Top up the gold set with items confirmed during live games, this cell does.
gold_path = PATHS.eval_dir / 'gold_set.jsonl'
items = harvest_gold_set(PATHS.runs_dir)

print(f'Harvested {len(items)} gold items from {PATHS.runs_dir}')
for cid, info in CATEGORIES.items():
    n = sum(1 for it in items if it.competition_id == cid)
    print(f'  {info.display_name:<35} {n:>4} items')

if items:
    save_gold_set(items, gold_path)
    print(f'\n✓ Gold set saved → {gold_path}')
else:
    print('\nNo items confirmed yet — play more live games first.')

### 2.1 Load the gold set

`GoldSet` is a chainable view over the gold items — every filter / sampler / splitter returns a new `GoldSet`, so you can branch experiments freely.

**Recipes:**

```python
full          = GoldSet.load(gold_path)          # everything
maths         = full.filter_category(Category.MATHS)
maths_hard    = maths.filter_level(min_level=11)
balanced      = full.take_per_level(3, seed=0)   # ≤3 per level 1..15
random_pilot  = full.sample(20, seed=42)         # 20 random items
train, test   = full.split(0.8, seed=42)         # 80/20 shuffled split
holdout       = full - test                      # set difference by question identity
```

Drop the resulting `GoldSet` straight into `evaluate_strategy(strategy, gold, ...)` — it's iterable and `__len__`-able.

In [ ]:
# Gold set, from data/eval/gold_set.jsonl we load. Build it first if missing.
gold_path = PATHS.eval_dir / 'gold_set.jsonl'

if not gold_path.exists():
    raise FileNotFoundError(
        f'Gold set not found at {gold_path}.\n'
        'Build it with one of:\n'
        '  python scripts/build_gold_set.py     (mines existing run logs)\n'
        'Or play games first to populate data/runs/, then re-build.'
    )

full = GoldSet.load(gold_path)
print(f'Full gold set: {len(full)} items')
full.print_stats()

### 2.1.1 Choose your eval subset

Pick **one** of the recipes below — or write your own — and assign the result to `gold`. Section 2.2 evaluates against whatever's in `gold`.

In [ ]:
# ─── Eval-subset selector. Edit me. ──────────────────────────────────

# (a) Default: full set, optionally capped to first N items.
gold = full if N_EVAL_QUESTIONS is None else full.take(N_EVAL_QUESTIONS)

# (b) Single-category — uncomment one.
# gold = full.filter_category(Category.MATHS)
# gold = full.filter_category(Category.SCIENCE)
# gold = full.filter_category(Category.HISTORY)
# gold = full.filter_category(Category.ENTERTAINMENT)

# (c) Difficulty slice.
# gold = full.filter_level(min_level=1,  max_level=5)      # easy tier
# gold = full.filter_level(min_level=6,  max_level=10)     # medium tier
# gold = full.filter_level(min_level=11, max_level=15)     # hard tier

# (d) Difficulty-balanced pilot — at most N per level.
# gold = full.take_per_level(3, seed=0)

# (e) Category-balanced pilot — at most N per category.
# gold = full.take_per_category(10, seed=0)

# (f) Random sample — reproducible with the seed.
# gold = full.sample(50, seed=42)

# (g) Train / held-out test.
# train, test = full.split(0.8, seed=42)
# gold = test     # evaluate on held-out only

# (h) Chain freely.
# gold = full.filter_category(Category.MATHS).filter_level(min_level=8).take_per_level(2, seed=0)

print(f'Eval subset: {len(gold)} items')
gold.print_stats()

### 2.2 Evaluate the strategy (offline)

In [ ]:
# Warm up once, evaluate, never warm again — this is the controlled comparison.
strategy.warm_up()

t0 = time.monotonic()
report: EvalReport = evaluate_strategy(strategy, gold, verbose=True)
print(f'\nTotal eval time: {time.monotonic() - t0:.1f}s')

report.print_summary()

### 2.3 Save the report

In [ ]:
# Persistent on disk, every report is. Section 3, the leaderboard, reads these.
# report_id was defined in Section 1.4 — see comment there.
report_path = save_report(report, name=report_id, eval_dir=PATHS.eval_dir)
print(f'Report saved as: {report_id}')

### 2.4 Per-category accuracy plot

In [ ]:
# Per-category accuracy heatmap, prefer over numbers we do.
cats = sorted(report.by_category.keys())
accs = [report.by_category[c].accuracy for c in cats]
ns   = [report.by_category[c].n        for c in cats]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(cats, accs, color='steelblue', edgecolor='white')
for bar, n in zip(bars, ns):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'n={n}', ha='center', fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title(f'{strategy.name}\noverall acc = {report.accuracy:.1%}')
ax.axhline(0.25, linestyle='--', color='grey', linewidth=1, label='random baseline (0.25)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Save the per-category numbers, separately for write-up tables.
df_cat = pd.DataFrame(
    [(c, s.n, s.correct, s.accuracy, s.ece) for c, s in report.by_category.items()],
    columns=['category', 'n', 'correct', 'accuracy', 'ece'],
)
save_results(df_cat, f'percategory__{report_id}')

### 2.5 Reliability diagram (calibration)

In [ ]:
# Calibration plot. Confidence vs accuracy buckets, overconfidence diagnose this does.
# Source: each EvalSample's confidence/correct, written by evaluate_strategy in-memory.
confidences = [s.confidence for s in report.samples]
corrects    = [s.correct    for s in report.samples]

from polimibot.eval.calibration import compute_calibration
cal = compute_calibration(confidences, corrects, n_bins=10)
plot_calibration(cal, title=f'Reliability — {strategy.name}',
                 output_path=PATHS.eval_dir / f'calibration__{report_id}.png')

### 2.6 Live games (optional)

In [ ]:
# Live play, opt-in this is. Skip if no client. Each game writes a JSONL run log.
RUN_LIVE_GAMES         = False                         # ← flip to True to play
GAMES_PER_COMPETITION  = 1
COMPS_TO_PLAY          = list(CATEGORIES.keys())       # all four; subset for shorter runs

if RUN_LIVE_GAMES:
    if client is None:
        raise RuntimeError('Set POLIMI_USER / POLIMI_PASS in env, then re-run Section 0.3.')
    results = play_session(
        client,
        competition_ids=COMPS_TO_PLAY,
        strategy=strategy,
        games_per_competition=GAMES_PER_COMPETITION,
        run_id=report_id,
        verbose=True,
    )
    df_live = pd.DataFrame([{
        'competition'   : r.competition_name,
        'final_level'   : r.final_level,
        'earned'        : r.earned_amount,
        'accuracy'      : r.accuracy,
        'elapsed_s'     : r.elapsed_seconds,
        'n_questions'   : r.n_questions,
    } for r in results])
    save_results(df_live, f'live__{report_id}')
    df_live
else:
    print('RUN_LIVE_GAMES=False — skipping live play.')

### 2.7 Retrieval diagnostic (optional, RAG only)

Recall@k tells you _whether the right article is in the top-k retrieved chunks_ — independent of whether the LLM then picks the right letter. Without it, you can't tell retrieval failures apart from generation failures.

**Workflow:**

1. **Bootstrap** a labeling stub from your gold set. Each row gets the retriever's current top-5 candidate article titles, so you can pick from a list rather than typing titles from memory.
2. **Label by hand**: open `data/eval/retrieval_gold.jsonl` in an editor, replace each `"gold_article_title": null` with the Wikipedia article title that should ideally be retrieved (or leave `null` to mean "no article suffices" — that row is skipped in scoring). 50 labels gets you a usable signal; 100+ is comfortable.
3. **Measure**: re-run the eval cell; recall@1/3/5/10 and per-category breakdown come out.

Skip this section entirely if `retriever is None` (you're not running RAG).

In [ ]:
# Retrieval diagnostic, optional. Skip if no RAG.
RETRIEVAL_GOLD_PATH = PATHS.eval_dir / 'retrieval_gold.jsonl'
RUN_RETRIEVAL_DIAGNOSTIC = True   # set False to skip

if not RUN_RETRIEVAL_DIAGNOSTIC or retriever is None:
    print('Retrieval diagnostic skipped (RUN_RETRIEVAL_DIAGNOSTIC=False or retriever=None).')
else:
    from polimibot.eval.retrieval import (
        build_labeling_template, save_retrieval_gold,
        load_retrieval_gold, evaluate_retrieval,
    )

    if not RETRIEVAL_GOLD_PATH.exists():
        # First time: emit a labeling stub. Open the JSONL and fill in titles.
        stub = build_labeling_template(full, retriever=retriever, k_candidates=5)
        save_retrieval_gold(stub, RETRIEVAL_GOLD_PATH)
        print(f'\nLabeling stub written → {RETRIEVAL_GOLD_PATH}')
        print('Open it, fill in "gold_article_title" for each row, then re-run this cell.')
    else:
        labeled = load_retrieval_gold(RETRIEVAL_GOLD_PATH)
        n_labeled = sum(1 for it in labeled if it.is_labeled)
        if n_labeled == 0:
            print(f'No labeled rows yet in {RETRIEVAL_GOLD_PATH}.')
            print('Edit the file and fill in "gold_article_title" for each question.')
        else:
            report = evaluate_retrieval(
                retriever, labeled,
                ks=(1, 3, 5, 10),
                use_category_filter=RAG_USE_CATEGORY_FILTER,
                use_reranker=RAG_USE_RERANKER,
                use_hybrid=RAG_USE_HYBRID,
                retriever_name=(
                    f'k={RAG_K}'
                    + ('+cat' if RAG_USE_CATEGORY_FILTER else '')
                    + ('+hybrid' if RAG_USE_HYBRID else '')
                    + ('+rerank' if RAG_USE_RERANKER else '')
                ),
            )
            report.print_summary()
            report.save(PATHS.eval_dir / f'retrieval__{report_id}.json')

> **Run observations** — _fill in after running this section._
>
> _What worked, what didn't, what would you try next?_

---

## 3. Compare

Read every saved report from `data/eval/*.json` into one comparison table. This is your leaderboard — it grows automatically as you save more reports in Section 2.

### 3.1 Build the leaderboard

In [ ]:
# Existing build_leaderboard reads every *.json in data/eval/, this it does.
from polimibot.eval.make_leaderboard import build_leaderboard

leaderboard = build_leaderboard(PATHS.eval_dir)
if leaderboard.empty:
    print('No reports yet. Run Section 2 with a strategy first.')
else:
    print(f'\nLeaderboard ({len(leaderboard)} strategies):')
leaderboard

### 3.2 Save the leaderboard

In [ ]:
# Persistent CSV, reproducible the table is.
if not leaderboard.empty:
    save_results(leaderboard, 'leaderboard')

### 3.3 Comparison plot

In [ ]:
# Strategies on the x-axis; accuracy bars sorted descending. Speak louder than tables, plots do.
if leaderboard.empty:
    print('Leaderboard empty — nothing to plot yet.')
else:
    fig, (ax_acc, ax_lat) = plt.subplots(1, 2, figsize=(13, 4))

    df_sorted = leaderboard.sort_values('accuracy', ascending=False)
    ax_acc.barh(df_sorted['strategy'], df_sorted['accuracy'], color='steelblue', edgecolor='white')
    ax_acc.set_xlim(0, 1.0)
    ax_acc.set_xlabel('Accuracy')
    ax_acc.invert_yaxis()
    ax_acc.axvline(0.25, linestyle='--', color='grey', linewidth=1, label='random')
    ax_acc.set_title('Accuracy by strategy')
    ax_acc.legend(loc='lower right')

    ax_lat.barh(df_sorted['strategy'], df_sorted['latency_p50_s'], color='salmon', edgecolor='white',
                label='p50')
    ax_lat.barh(df_sorted['strategy'], df_sorted['latency_p95_s'] - df_sorted['latency_p50_s'],
                left=df_sorted['latency_p50_s'], color='lightsalmon', edgecolor='white',
                label='p50→p95')
    ax_lat.invert_yaxis()
    ax_lat.set_xlabel('Seconds per question')
    ax_lat.set_title('Latency (p50 + p95 tail)')
    ax_lat.legend(loc='lower right')

    plt.tight_layout()
    plt.show()

### 3.4 Per-category cross-strategy heatmap

In [ ]:
# Reports include per-category accuracy. Heatmap, build it from disk we do.
def _load_report(p: Path) -> dict:
    return json.loads(p.read_text())

rows = []
for p in sorted(PATHS.eval_dir.glob('*.json')):
    data = _load_report(p)
    for cat, stats in data.get('by_category', {}).items():
        rows.append({
            'strategy': data['strategy_name'],
            'category': cat,
            'accuracy': stats.get('accuracy', float('nan')),
            'n':        stats.get('n', 0),
        })

if rows:
    heat = (pd.DataFrame(rows)
              .pivot_table(index='strategy', columns='category', values='accuracy'))
    fig, ax = plt.subplots(figsize=(7, max(2, 0.5 * len(heat))))
    im = ax.imshow(heat.values, vmin=0, vmax=1, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels(heat.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(heat.index)));   ax.set_yticklabels(heat.index)
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            if pd.notna(v):
                ax.text(j, i, f'{v:.0%}', ha='center', va='center',
                        color='white' if v > 0.55 else 'black', fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    ax.set_title('Per-category accuracy across strategies')
    plt.tight_layout()
    plt.show()
    save_results(heat.reset_index(), 'percategory_matrix')
else:
    print('No per-category data yet.')

> **Comparison observations** — _fill in after running this section._
>
> _What worked, what didn't, what would you try next?_

---

## 4. Save

Consolidated artefacts for the write-up. All previous sections write incrementally; this section just summarises what's on disk.

### 4.1 Inventory

In [ ]:
# What did we produce? List, count, recap.
def _list(dir_: Path, pattern: str) -> list[Path]:
    return sorted(dir_.glob(pattern)) if dir_.exists() else []

run_logs    = _list(PATHS.runs_dir,           '*.jsonl')
eval_jsons  = _list(PATHS.eval_dir,           '*.json')
result_csvs = _list(RESULTS_DIR,              '*.csv') if RESULTS_DIR.exists() else []
plots       = _list(PATHS.eval_dir,           '*.png')

inventory = pd.DataFrame({
    'kind':  ['run logs (JSONL)', 'eval reports (JSON)', 'results (CSV)', 'plots (PNG)'],
    'count': [len(run_logs), len(eval_jsons), len(result_csvs), len(plots)],
})
print(inventory.to_string(index=False))

### 4.2 Final summary

In [ ]:
# A concise text summary, useful to paste into the report it is.
print('═' * 60)
print(f'PoliMillionaire — experiment summary')
print('═' * 60)
print(f'API URL          : {RUNTIME.api_url}')
print(f'Model            : {MODEL_ID}  (mock={USE_MOCK})')
print(f'Strategy         : {strategy.name}')
if not leaderboard.empty:
    top = leaderboard.iloc[0]
    print(f"Leader strategy  : {top['strategy']}")
    print(f"Leader accuracy  : {top['accuracy']:.1%}")
    print(f"Leader p50/p95   : {top['latency_p50_s']:.2f}s / {top['latency_p95_s']:.2f}s")
print(f'Reports on disk  : {len(eval_jsons)}')
print('═' * 60)

> **Save / final observations** — _fill in after running this section._
>
> _What worked, what didn't, what would you try next?_

---

## Appendix — VRAM hygiene

When you finish a model and want to load a different one, run the cell below first. T4 GPUs in Colab die without warning if VRAM gets fragmented.

In [ ]:
# Free the current LLM completely. The next Section 1.2 run will reload from scratch.
if 'llm' in globals():
    unload_llm(llm)
    del llm
    if 'LOADED_MODEL_ID' in globals():
        del LOADED_MODEL_ID
clear_vram()
print('VRAM cleared. Edit MODEL_ID in Section 1.1 and re-run Sections 1.2 → 2.')